# Introduzione a Pytorch
Pipeline di base:
- Definizione del dataset
- Definizione del modello
- Definizione del ciclo di addestramento


## Definizione del dataset
- Lettura dei dati
- Split per training, validation e test set
- Creazione del **DataLoader** per ciclare sul dataset

In [ ]:
# Lettura dei dati (base)
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data.dataset import random_split

# Data augmentation
data_augment = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor()
])

# Lettura dei dati
train_dataset = ImageFolder(root='path/to/dataset', transform=data_augment)
train_dst, valid_dst = random_split(train_dataset, lengths=[50000, 5000])


In [ ]:
# Lettura dati (avanzati)
import os, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class SegmentDataLoader(Dataset):
  # Costruttore con tutti i path, le liste di dati, le trasformazioni di default, ...
  def __init__(self, root1, root2, root3, mask_folder, img_size, transform):
    self.img_folderC = root1
    self.img_folderM01 = root2
    self.img_folderM02 = root3
    self.mask_folder = mask_folder
    self.transform = transform
    self.img_size = img_size

  transform2 = transforms.Compose([
      transforms.Resize((self.img_size, self.img_size)),
      transforms.ToTensor(),
      transforms.Grayscale()
  ])

  self.transform2 = transform2

  self.folders, self.images = [], []
  self.imagesC = os.listdir(self.img_folderC)
  self.imgesM01 = os.listdir(self.img_folderM01)
  self.imgesM02 = os.listdir(self.img_folderM02)
  self.labels = os.listdir(self.mask_folder)

  # Per ottenere la dimensione del dataset
  def __len__(self):
    return len(self.imagesC)

  # Per ottenere il dato dal dataset da usare nella fase di addestramento
  def __getitem__(self, index):
    imgC = os.path.join(self.img_folderC, self.imagesC(index))
    imgM01 = os.path.join(self.img_folderM01, self.imgesM01(index))
    imgM02 = os.path.join(self.img_folderM02, self.imgesM02(index))
    label = os.path.join(self.mask_folder, self.labels(index))

    img_c = self.transform(Image.open(imgC).convert('RGB'))
    img_m1 = self.transform(Image.open(imgM01).convert('RGB'))
    img_m2 = self.transform(Image.open(imgM02).convert('RGB'))
    label_t = self.transform(Image.open(label).convert('RGB'))

    return {'c': img_c, 'm1': img_m1, 'm2': img_m2, 'label': label_t}

Il DataLoader è la classe di torch che permette di caricare e usare un dataset in modo iterabile. Consente, sui dati letti, di fare:
- Batch di dati
- Shuffling
- Caricamento parallelo di dati
- Data Augmentation

In [ ]:
# Creazione del DataLoader
from  torch.utils.data import DataLoader

dataloader = DataLoader(train_dst, batch_size=32, shuffle=True)

## Definizione del modello

In [ ]:
# Definizione del modello (base)
import torch

class FirstCNN(torch.nn.Module):
  # Struttura della rete nel costruttore della classe
  def __init__(self, num_classes):
    super(FirstCNN, self).__init__()

    self.conv1 = torch.nn.Conv2d(in_channel=3, out_channel=32)
    self.conv2 = torch.nn.Conv2d(in_channel=32, out_channel=64)
    self.conv3 = torch.nn.Conv2d(in_channel=64, out_channel=128)

    self.fullyC = torch.nn.Linear(..., num_classes)
    self.activation = torch.nn.ReLU()

  # Processo di forward della rete
  def forward(self, input_data):
    x = self.conv1(input_data)
    x = self.conv2(x)
    x = self.conv3(x)
    ...
    output = self.fullyC(x)
    return output

In [ ]:
# Definizione del modello (avanzato)
import torch
from torch import nn

class Encoder(nn.Module):
  def __init__(self, in_channels=3, base_width=64, tiler=None):
    # Encoder 1
    self.m1_eb1 = nn.Sequential(
        nn.Conv2d(in_channels, base_width, kernel_size=5, stride=1, padding=2, bias=False),
        nn.BatchNorm2d(base_width),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m1_eb2 = nn.Sequential(
        nn.Conv2d(base_width, base_width*2, kernel_size=5, stride=1, padding=2, bias=False),
        nn.GELU(),
        nn.Conv2d(base_width*2, base_width*4, kernel_size=5, stride=1, padding=2, bias=False),
        nn.BatchNorm2d(base_width*4),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m1_eb3 = nn.Sequential(
        nn.Conv2d(base_width*4, base_width*8, kernel_size=3, stride=1, padding=1, bias=False),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m1_eb4 = nn.Sequential(
        nn.Conv2d(base_width*8, base_width*8, kernel_size=3, stride=1, padding=1, bias=False)
    )
    # Encoder 2
    self.m2_eb1 = nn.Sequential(
        nn.Conv2d(in_channels, base_width, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(base_width),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m2_eb2 = nn.Sequential(
        nn.Conv2d(base_width, base_width*2, kernel_size=3, stride=1, padding=1, bias=False),
        nn.GELU(),
        nn.Conv2d(base_width*2, base_width*4, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(base_width*4),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m1_eb3 = nn.Sequential(
        nn.Conv2d(base_width*4, base_width*8, kernel_size=3, stride=1, padding=1, bias=False),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.m1_eb4 = nn.Sequential(
        nn.Conv2d(base_width*8, base_width*8, kernel_size=1, stride=1, padding=0, bias=False)
    )

  def forward(self, x1, x2):
    # Encoder 1
    m1_s1 = self.m1_eb1(x1)
    m1_s2 = self.m1_eb2(m1_s1)
    m1_s3 = self.m1_eb3(m1_s2)
    m1_ls = self.m1_eb4(m1_s3)
    # Encoder 2
    m2_s1 = self.m2_eb1(x2)
    m2_s2 = self.m2_eb2(m2_s1)
    m2_s3 = self.m2_eb3(m2_s2)
    m2_ls = self.m2_eb4(m2_s3)
    # Fusione
    lsc = torch.cat([m1_ls, m2_ls], dim=1)
    ls = self.combo_latent(lsc)
    return ls


## Definizione del loop di addestramento
- Istanziare il modello
- Spostarlo sulla GPU
- Istanziare l'ottimizzatore
- Definizione della loss
- Loop **for** per ciclare sui dati

In [ ]:
# Definizione del loop di addestramento
model = FirstCNN(num_classes=10)          # Istanzio il modello
model = model.cuda()                      # Sposto il modello sulla GPU
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
CE_loss = torch.nn.CrossEntropyLoss()     # Definisco la loss

for epoch in range(num_epochs):
  model.train()                           # Modello in modalità di training

  for batch in train_dataloader:
    img = batch[0].cuda()                 # Sposto i dati sulla GPU
    labels = batch[1].cuda()

    # Forward pass
    output = model(img)                   # Faccio passare i dati attraverso la rete
    loss = CE_loss(img, labels)           # Calcolo la loss

    # Backpropagation
    optimizer.zero_grad()                 # Azzero i gradienti per non accumularli
    loss.backward()                       # Calcolo i gradienti
    optimizer.step()                      # Aggiorno i pesi

  model.eval()
  for vbatch in valid_dataloader:
    img = batch[0].cuda()
    labels = batch[1].cuda()

    # Forward pass
    output = model(img)
    loss = CE_loss(img, labels)